# L3: DaughterNet from scratch on the dorado pseudo-labels (Colab T4)

Trains the 3-class DaughterNet (`cont / parent / daughter`, DivNet UNet-GAP body, input `(5, 16, 32, 32)`) on the
pseudo-labelled crops of the public Ultrack zebrafish ("dorado") embryo. **No Kaggle label is used**, so the laptop can
later judge the model on all Kaggle GT labels (`tools/eval_daughternet.py eval`).

**Before running**
1. *Runtime > Change runtime type > T4 GPU*.
2. Upload to `/content` (Files pane on the left, or cell 3, or put them on Google Drive and use cell 2):
   - `divnet_infer.py`, `daughternet.py`, `train_dorado.py` (from the repo's `tools/`);
   - the shard files `b<block>_z<k>.npz` from `data_external/dorado/shards/` (about 1.4 GB for 8 shards), loose or
     in one `.zip`. Keep block 417: it is the early-stopping hold-out;
   - optional, strict mode only: `kaggle44b6_in_dorado.json` from `data_external/dorado/`. The Kaggle 44b6 videos
     are crops of this very recording (`tools/register_44b6_dorado.py`). By default all dorado samples are used and
     the laptop evaluator reports overlap-free `44b6_clean` rows (6bba is another embryo). With `EXCLUDE_44B6 = True`
     (cell 5) the trainer drops every dorado sample inside a 44b6 clip (39% of the samples).
3. *Runtime > Run all*. Output: `/content/dorado_only.pt` (`{'model_state', 'config'}`), `dorado_only.history.json`,
   `train_dorado.log`; the last cell downloads them.

Expected time on a T4: roughly 10 min (6-20 min) for ~12k samples x 40 epochs (~5.4 GFLOP per training sample);
after epoch 0 the log prints the measured samples/s.

In [ ]:
# 1. GPU check
import sys, torch
print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU (CPU training takes many hours)')

In [ ]:
# 2. Optional: files on Google Drive (faster than a browser upload for ~1.4 GB of shards)
USE_DRIVE = False
DRIVE_DIR = '/content/drive/MyDrive/dorado'   # folder with the b*_z*.npz shards and/or the 3 .py files (or a .zip)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# 3. Optional: upload from this cell instead of the Files pane (select the 3 .py files and the shards / a zip)
UPLOAD = False
if UPLOAD:
    from google.colab import files
    files.upload()   # saved into /content

In [ ]:
# 4. Put the 3 python files into a `tools` package and gather the shards into /content/shards
import os, shutil, zipfile
from pathlib import Path
import numpy as np

C = Path('/content')
drive_dir = Path(DRIVE_DIR) if USE_DRIVE else None
for d in [C] + ([drive_dir] if drive_dir else []):
    for z in sorted(d.glob('*.zip')):
        dest = C / 'unzipped' / z.stem
        if not dest.exists():
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
            print('unzipped', z, '->', dest)


def find(pattern):
    # /content itself flat (never walk /content/drive or sample_data); unzipped/ and DRIVE_DIR recursively
    hits = sorted(C.glob(pattern))
    for d in [C / 'unzipped'] + ([drive_dir] if drive_dir else []):
        if d.exists():
            hits += sorted(d.rglob(pattern))
    return hits


pkg = C / 'tools'
pkg.mkdir(exist_ok=True)
(pkg / '__init__.py').touch()
for name in ('divnet_infer.py', 'daughternet.py', 'train_dorado.py'):
    hits = find(name)
    if hits:
        shutil.copy(hits[0], pkg / name)
        print('tools/' + name, '<-', hits[0])
    elif not (pkg / name).exists():
        raise FileNotFoundError(f'upload {name} (from the repo tools/) to /content')

shard_dir = C / 'shards'
shard_dir.mkdir(exist_ok=True)
for p in find('b*_z*.npz'):
    if p.name.endswith('.tmp.npz') or p.parent == shard_dir:
        continue
    dst = shard_dir / p.name
    if not dst.exists():
        os.symlink(p, dst)
hits = find('kaggle44b6_in_dorado.json')          # only used when EXCLUDE_44B6 = True (cell 5)
if hits:
    shutil.copy(hits[0], shard_dir / 'kaggle44b6_in_dorado.json')
    print('Kaggle 44b6 registration <-', hits[0])
shards = sorted(shard_dir.glob('b*_z*.npz'))
if not shards:
    raise FileNotFoundError('no b*_z*.npz shard found: upload them to /content (or DRIVE_DIR)')
tot = np.zeros(3, int)
blocks = set()
for p in shards:
    with np.load(p) as z:
        y = z['y']
    c = np.bincount(y, minlength=3)
    tot += c
    blocks.add(int(p.name[1:4]))
    print(f'{p.name}: {len(y)} samples cont/parent/daughter {c.tolist()} ({p.stat().st_size / 1e6:.0f} MB)')
print('total', int(tot.sum()), tot.tolist(), 'blocks', sorted(blocks))
if 417 not in blocks:
    print('WARNING: no block 417 shard -> a random lineage-grouped 15% hold-out is used for early stopping')

In [ ]:
# 5. Train (tools/train_dorado.py: AdamW 1e-3 cosine, batch 128, class-balanced sampler, flips + yx rot90,
#    Kaggle clip-border padding p=auto (~0.8, matches the Kaggle 61%), gain 0.8-1.2, label smoothing 0.1;
#    hold-out block 417; the epoch with the best val macro CE is saved)
# Run as a subprocess (not `!... | tee`, which hides the exit code): a failed run raises here and stops Run all, and
# outputs of an earlier run are moved to *.prev first so cell 6 can never validate a stale checkpoint.
import os, subprocess, sys
EPOCHS = 40
EXCLUDE_44B6 = False   # True: strict mode, drop dorado samples inside the Kaggle 44b6 clips (needs the json, cell 4)
OUT = '/content/dorado_only.pt'
for f in (OUT, '/content/dorado_only.history.json', '/content/dorado_only.last.pt'):
    if os.path.exists(f):
        os.replace(f, f + '.prev')
cmd = [sys.executable, '-u', '-m', 'tools.train_dorado', '--shards', '/content/shards', '--out', OUT,
       '--epochs', str(EPOCHS), '--batch', '128', '--save-last']
if EXCLUDE_44B6:
    reg = '/content/shards/kaggle44b6_in_dorado.json'
    if not os.path.exists(reg):
        raise FileNotFoundError('EXCLUDE_44B6 needs kaggle44b6_in_dorado.json (upload it, rerun cell 4)')
    cmd += ['--exclude-kaggle', reg]
print(' '.join(cmd))
with open('/content/train_dorado.log', 'w') as log_fh:
    proc = subprocess.Popen(cmd, cwd='/content', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
        log_fh.write(line)
    rc = proc.wait()
if rc != 0 or not os.path.exists(OUT):
    raise RuntimeError(f'training failed (exit code {rc}): see /content/train_dorado.log')

In [ ]:
# 6. Check: the checkpoint loads strictly through tools.daughternet.load_daughternet; summary of the run
import json, sys
sys.path.insert(0, '/content')
from tools.daughternet import CLASSES, load_daughternet
b = load_daughternet(['/content/dorado_only.pt'], device='cpu')
cfg = b['config']
t = cfg['train']
assert tuple(cfg['classes']) == tuple(CLASSES) == ('cont', 'parent', 'daughter')
print('kaggle labels used:', t['kaggle_labels_used'], '| shards', t['shards'])
print('Kaggle 44b6 leak-free:', t.get('kaggle44b6_leak_free'), '| exclusion', t.get('kaggle44b6_exclusion'))
print('train', t['n_train'], t['train_counts'], '| val', t['n_val'], t['val_counts'], '|', t['val'])
print('best epoch', t['best_epoch'], 'val macro CE', round(t['best_val_macro_ce'], 4))
print({k: (None if v is None else round(v, 3)) for k, v in t['best_val_metrics'].items() if k.startswith('auc')})
r3 = lambda v: None if v is None else round(v, 3)
for h in json.load(open('/content/dorado_only.history.json')):
    print(h['epoch'], 'val', round(h['val_macro_ce'], 4), 'auc parent', r3(h['val_auc_parent']),
          'daughter', r3(h['val_auc_daughter']), 'vs hard', r3(h['val_auc_parent_vs_hard']),
          r3(h['val_auc_daughter_vs_hard']), h['train_samples_per_s'], 'samples/s')

In [ ]:
# 7. Download (and copy to Drive when mounted)
import os, shutil
from google.colab import files
outs = ['/content/dorado_only.pt', '/content/dorado_only.history.json', '/content/train_dorado.log']
if USE_DRIVE:
    for f in outs:
        if os.path.exists(f):
            shutil.copy(f, DRIVE_DIR)
            print('copied', f, '->', DRIVE_DIR)
for f in outs:
    if os.path.exists(f):
        files.download(f)

**Next, on the laptop** (put the checkpoint into `data_external/l3/`):

    python tools/eval_daughternet.py eval --ckpt data_external/l3/dorado_only.pt --out-csv data_external/l3/dorado_only_kaggle_p.csv
    python tools/eval_daughternet.py eval --ckpt data_external/l3/dorado_only.pt --oof <kdn-train output>/oof.csv

The Kaggle evaluation cache (`data_external/l2/kaggle_eval_crops.npz`, all 199 videos) is built by
`python tools/eval_daughternet.py build`.